# `crm_prd_info` Exploratory Data Analysis

## Init

In [0]:
from pyspark.sql.functions import col, length, trim

## Reading from Bronze layer

In [0]:
df = spark.table("data_lakehouse.bronze.crm_prd_info")

## Exploring the data

### `prd_key` column contains category_id and product_key concatenated

In [0]:
df.select(col("prd_key")).show(10)

### Column with null values

In [0]:
df.where(col("prd_cost").isNull()).show()

### Leading and trailing spaces

In [0]:
df.where(length(col("prd_line")) != length(trim(col("prd_line")))).show()

### Non-descriptive values

In [0]:
df.select(col("prd_line")).distinct().show()

### Slowly Changing Dimension - Type 2

In [0]:
df.orderBy(col("prd_key"), col("prd_start_dt")).show(50)

> `null` indicates current `prd_cost` value

### Incorrect date values

In [0]:
df.where(col("prd_start_dt") > col("prd_end_dt")).show()

## Issues found

1. Column: `prd_key` contains product key and category id concatenated.
2. Column: `prd_cost` has null values.
3. Column: `prd_line` has leading and trailing spaces.
4. Column: `prd_line` does not have descriptive values.
5. Columns: `prd_start_dt` and `prd_end_dt` have incorrect date values.
6. Column: `prd_end_dt` uses `null` to represent the current `prd_cost`. Use instead a centinel value.
7. Add a column `is_current` to represent in a better way the current `prd_cost` using the date columns.
8. Columns must be casted to their respective data types (e.g. `prd_id` to `bigint`, `prd_start_dt` to `date`).
9. Column names do not follow any naming convention (e.g. `prd_id` instead of `product_id`, `prd_end_dt` instead of `end_date`).